# Music Annotation Project V7.5: Exact Geometry Preservation and Surrounding-Area Evidence

**Runtime:** CPU. Select **Runtime > Run all**.

V7.5 benchmarks exact preservation of observable printed geometry inside an existing annotation-removal mask. It also records surrounding-area evidence for a future two-axis framework:

- **annotation confidence**: later supplied by YOLO and other annotation evidence;
- **correction safety**: later estimated from local musical structure and reconstruction risk.

This notebook does **not** calculate a combined score or auto-accept any correction. It records evidence for later validation.

## Routing rule

- confidently supported printed geometry: copy the observed pixels byte-for-byte;
- confidently light non-structural pixels: replace with local paper;
- ambiguous pixels: permit Telea only here;
- outside the removal mask: preserve byte-for-byte.

Clean targets are never supplied to reconstruction. They are used only afterwards for metrics and evidence diagnostics.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


In [ ]:
import cv2, json, hashlib, shutil, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.metrics import structural_similarity as ssim

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'; OBSERVED_DIR=ROOT/'aligned'; MASK_DIR=ROOT/'masks'/'corrected'
V71_ROOT=ROOT/'reconstruction_benchmark_v7_1_real'; V71_STATE=V71_ROOT/'benchmark_state.json'
OUTPUT_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'; RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v7_5'
OUT=OUTPUT_ROOT/'runs'/RUN; TABLES=OUT/'tables'; MAPS=OUT/'classification_maps'; CONTACTS=OUT/'contact_sheets'; CACHE=OUTPUT_ROOT/'persistent_cache'
for d in [OUT,TABLES,MAPS,CONTACTS,CACHE]:d.mkdir(parents=True,exist_ok=True)
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
EXCLUDE_PAGES={'Page201'}; DILATION=2; RAW_THRESHOLDS=[80,110]
PAPER_THRESHOLD=155; NORMALISED_PAPER_THRESHOLD=165; BACKGROUND_SIGMA=75.0
HORIZONTAL_LENGTH=35; VERTICAL_LENGTH=9; OUTSIDE_SUPPORT_RADIUS=13; GEOMETRY_GROW=1
COMPACT_MIN_AREA=6; COMPACT_MAX_AREA=500; COMPACT_MAX_DIMENSION=42; COMPACT_MIN_FILL=.16
RING_RADIUS=18; DARK_CONTEXT_THRESHOLD=170; EDGE_LOW=70; EDGE_HIGH=170
METHODS=['telea','hard_raw_t80','hard_raw_t110','geometry_t80','geometry_t110']
FORCE_REBUILD=False

def image_files(folder):return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if Path(folder).exists() else []
def maps(folder):return {p.stem:p for p in image_files(folder)}
def md5(path):
 h=hashlib.md5()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def objhash(x):return hashlib.md5(json.dumps(x,sort_keys=True).encode()).hexdigest()
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0];assert np.array_equal(out[mask==0],original[mask==0]);return out

def load_state():
 state=json.loads(V71_STATE.read_text())
 if not isinstance(state.get('items'),dict):raise RuntimeError('Invalid V7.1 state')
 return state
state=load_state(); clean_map,observed_map,mask_map=maps(CLEAN_DIR),maps(OBSERVED_DIR),maps(MASK_DIR)
pages=[];excluded=[];issues=[]
for key,item in sorted(state['items'].items()):
 if int(item.get('dilation',-1))!=DILATION:continue
 page=item.get('page');cache=Path(item.get('reconstruction_cache',''))
 if page in EXCLUDE_PAGES:excluded.append({'page':page,'reason':'explicit exclusion'});continue
 if page not in clean_map or page not in observed_map or page not in mask_map:issues.append({'page':page,'reason':'missing input'});continue
 corrected=cv2.imread(str(mask_map[page]),0)
 if corrected is None or not np.any(corrected>0):excluded.append({'page':page,'reason':'empty/unreadable corrected mask'});continue
 try:
  with np.load(cache) as z:ok={'mask','telea'}.issubset(z.files)
 except:ok=False
 if not item.get('reconstruction_complete') or not ok:issues.append({'page':page,'reason':'V7.1 cache unavailable'});continue
 pages.append(page)
pages=sorted(set(pages));pd.DataFrame(excluded).to_csv(TABLES/'excluded_pages.csv',index=False);pd.DataFrame(issues).to_csv(TABLES/'discovery_issues.csv',index=False)
print('Eligible pages:',len(pages),'Excluded:',len(excluded),'Issues:',len(issues))


## Reconstruction classifier and evidence helpers


In [ ]:
def background_normalise(image):
 bg=cv2.GaussianBlur(image,(0,0),BACKGROUND_SIGMA);return cv2.divide(image,np.maximum(bg,1),scale=255)
def paper_map(observed,mask):
 out=np.full_like(observed,255);n,lab,_,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
 for i in range(1,n):
  comp=lab==i;ring=(cv2.dilate(comp.astype(np.uint8),np.ones((21,21),np.uint8))>0)&~comp
  vals=observed[ring&(observed>=180)];out[comp]=int(np.median(vals)) if vals.size else 255
 return out

def supported_horizontal(observed,mask,t):
 dark=observed<t;outside=dark&(mask==0)
 out_lines=cv2.morphologyEx(outside.astype(np.uint8),cv2.MORPH_OPEN,np.ones((1,HORIZONTAL_LENGTH),np.uint8))>0
 all_lines=cv2.morphologyEx(dark.astype(np.uint8),cv2.MORPH_OPEN,np.ones((1,HORIZONTAL_LENGTH),np.uint8))>0
 support=cv2.dilate(out_lines.astype(np.uint8),np.ones((3,2*OUTSIDE_SUPPORT_RADIUS+1),np.uint8))>0
 return all_lines&support&(mask>0)
def supported_vertical(observed,mask,t):
 dark=observed<t;outside=dark&(mask==0)
 out_lines=cv2.morphologyEx(outside.astype(np.uint8),cv2.MORPH_OPEN,np.ones((VERTICAL_LENGTH,1),np.uint8))>0
 all_lines=cv2.morphologyEx(dark.astype(np.uint8),cv2.MORPH_OPEN,np.ones((VERTICAL_LENGTH,1),np.uint8))>0
 support=cv2.dilate(out_lines.astype(np.uint8),np.ones((2*OUTSIDE_SUPPORT_RADIUS+1,3),np.uint8))>0
 return all_lines&support&(mask>0)
def supported_compact(observed,mask,t):
 dark=observed<t;n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);result=np.zeros_like(dark)
 outside_near=cv2.dilate((dark&(mask==0)).astype(np.uint8),np.ones((5,5),np.uint8))>0
 for i in range(1,n):
  x,y,w,h,a=map(int,stats[i]);fill=a/max(w*h,1);comp=lab==i
  if COMPACT_MIN_AREA<=a<=COMPACT_MAX_AREA and max(w,h)<=COMPACT_MAX_DIMENSION and fill>=COMPACT_MIN_FILL and np.any(comp&(mask>0)) and np.any(comp&outside_near):result|=comp&(mask>0)
 return result

def geometry_evidence(observed,mask,t):
 h=supported_horizontal(observed,mask,t);v=supported_vertical(observed,mask,t);c=supported_compact(observed,mask,t);g=h|v|c
 if GEOMETRY_GROW:
  g=(cv2.dilate(g.astype(np.uint8),np.ones((3,3),np.uint8))>0)&(observed<t)&(mask>0)
 return g,h,v,c

def geometry_candidate(observed,mask,telea,paper,norm,t):
 preserve,h,v,c=geometry_evidence(observed,mask,t)
 remove=(mask>0)&~preserve&(observed>=PAPER_THRESHOLD)&(norm>=NORMALISED_PAPER_THRESHOLD)
 ambiguous=(mask>0)&~preserve&~remove
 out=observed.copy();out[remove]=paper[remove];out[ambiguous]=telea[ambiguous];out[preserve]=observed[preserve]
 assert np.array_equal(out[preserve],observed[preserve])
 return immutable(observed,out,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':h,'vertical':v,'compact':c}
def hard_raw(observed,mask,paper,t):
 out=observed.copy();q=(mask>0)&(observed>=t);out[q]=paper[q];return immutable(observed,out,mask)

def component_context_evidence(observed,mask,comp,routes,primitive_maps,component_id,stats_row):
 kernel=np.ones((2*RING_RADIUS+1,2*RING_RADIUS+1),np.uint8)
 ring=(cv2.dilate(comp.astype(np.uint8),kernel)>0)&~comp
 page=np.ones_like(comp,dtype=bool);context=ring&page
 edges=cv2.Canny(observed,EDGE_LOW,EDGE_HIGH)>0;dark=observed<DARK_CONTEXT_THRESHOLD
 dist_to_horizontal=cv2.distanceTransform((~primitive_maps['horizontal']).astype(np.uint8),cv2.DIST_L2,5)
 area=int(comp.sum());x,y,w,h,a=map(int,stats_row);bbox=max(w*h,1)
 mask_depth=cv2.distanceTransform((mask>0).astype(np.uint8),cv2.DIST_L2,5)
 def frac(q):return float(np.sum(q&comp)/max(area,1))
 return {
  'component':int(component_id),'x':x,'y':y,'w':w,'h':h,'area':area,'aspect_ratio':float(w/max(h,1)),'fill_ratio':float(area/bbox),
  'touches_page_border':bool(x==0 or y==0 or x+w>=observed.shape[1] or y+h>=observed.shape[0]),
  'context_pixels':int(context.sum()),'context_edge_density':float(np.mean(edges[context])) if context.any() else np.nan,
  'context_dark_density':float(np.mean(dark[context])) if context.any() else np.nan,
  'context_mean_intensity':float(np.mean(observed[context])) if context.any() else np.nan,
  'context_intensity_std':float(np.std(observed[context])) if context.any() else np.nan,
  'minimum_distance_to_supported_horizontal':float(np.min(dist_to_horizontal[comp])) if area else np.nan,
  'maximum_mask_depth':float(np.max(mask_depth[comp])) if area else np.nan,
  'horizontal_fraction':frac(primitive_maps['horizontal']),'vertical_fraction':frac(primitive_maps['vertical']),'compact_fraction':frac(primitive_maps['compact']),
  'preserve_fraction':frac(routes['preserve']),'remove_fraction':frac(routes['remove']),'ambiguous_fraction':frac(routes['ambiguous']),
  'reconstruction_dependency_fraction':frac(routes['ambiguous'])
 }


## Generate candidates and record surrounding-area evidence


In [ ]:
cache_rows=[];routing_rows=[];evidence_rows=[];truth_rows=[]
for ix,page in enumerate(pages,1):
 clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);item=state['items'][f'{page}|d{DILATION}']
 with np.load(item['reconstruction_cache']) as z:mask=z['mask'];telea=z['telea']
 paper=paper_map(observed,mask);norm=background_normalise(observed)
 outputs={'telea':telea,'hard_raw_t80':hard_raw(observed,mask,paper,80),'hard_raw_t110':hard_raw(observed,mask,paper,110)};route_sets={}
 for t in RAW_THRESHOLDS:
  candidate,routes=geometry_candidate(observed,mask,telea,paper,norm,t);outputs[f'geometry_t{t}']=candidate;route_sets[t]=routes
  m=mask>0;routing_rows.append({'page':page,'method':f'geometry_t{t}','mask_pixels':int(m.sum()),**{f'{k}_pixels':int(routes[k].sum()) for k in ['preserve','remove','ambiguous']},**{f'{k}_fraction':float(routes[k].sum()/max(m.sum(),1)) for k in ['preserve','remove','ambiguous']}})
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
 clean_ink=cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
 for component in range(1,n):
  if stats[component,cv2.CC_STAT_AREA]<20:continue
  comp=lab==component
  for t in RAW_THRESHOLDS:
   routes=route_sets[t];primitives={k:routes[k] for k in ['horizontal','vertical','compact']}
   row={'page':page,'method':f'geometry_t{t}',**component_context_evidence(observed,mask,comp,routes,primitives,component,stats[component])}
   evidence_rows.append(row)
   truth_rows.append({'page':page,'method':f'geometry_t{t}','component':component,'clean_print_fraction':float(np.mean(clean_ink[comp])),'clean_paper_fraction':float(np.mean(~clean_ink[comp])),
    'preserved_clean_print_precision':float(np.sum(routes['preserve']&comp&clean_ink)/max(np.sum(routes['preserve']&comp),1)),
    'removed_clean_paper_precision':float(np.sum(routes['remove']&comp&~clean_ink)/max(np.sum(routes['remove']&comp),1)),
    'ambiguous_clean_print_fraction':float(np.sum(routes['ambiguous']&comp&clean_ink)/max(np.sum(routes['ambiguous']&comp),1))})
 fingerprint=objhash({'page':page,'observed':md5(observed_map[page]),'mask':md5(mask_map[page]),'schema':'v7.5-surrounding-evidence-1'})
 cache_path=CACHE/f'{page}_{fingerprint}.npz';np.savez_compressed(cache_path,mask=mask,**outputs,**{f'g{t}_{k}':v.astype(np.uint8) for t,rs in route_sets.items() for k,v in rs.items()})
 cache_rows.append({'page':page,'cache':str(cache_path),'fingerprint':fingerprint});print(ix,'/',len(pages),page)
pd.DataFrame(cache_rows).to_csv(TABLES/'cache_status.csv',index=False)
pd.DataFrame(routing_rows).to_csv(TABLES/'routing_by_page.csv',index=False)
evidence=pd.DataFrame(evidence_rows);evidence.to_csv(TABLES/'surrounding_area_evidence_by_component.csv',index=False)
truth=pd.DataFrame(truth_rows);truth.to_csv(TABLES/'surrounding_evidence_truth_diagnostics.csv',index=False)
summary=evidence.groupby('method',as_index=False).agg(components=('component','count'),mean_context_edge_density=('context_edge_density','mean'),mean_context_dark_density=('context_dark_density','mean'),mean_distance_to_staff=('minimum_distance_to_supported_horizontal','mean'),mean_mask_depth=('maximum_mask_depth','mean'),mean_preserve_fraction=('preserve_fraction','mean'),mean_remove_fraction=('remove_fraction','mean'),mean_ambiguous_fraction=('ambiguous_fraction','mean'),mean_reconstruction_dependency=('reconstruction_dependency_fraction','mean'))
summary.to_csv(TABLES/'surrounding_area_evidence_summary.csv',index=False);display(summary)


## Benchmark metrics and route diagnostics


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def classify(clean,mask):
 ink=printed_ink(clean);long=cv2.morphologyEx(ink.astype(np.uint8),cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0;near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0;m=mask>0
 return {'all':m,'background':m&~ink,'staff':m&staff,'symbol':m&ink&~near,'mixed':m&ink&near}
def tol_f1(ref,pred,region,tol=1):
 ref=ref&region;pred=pred&region
 if not ref.sum():return np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8);precision=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1);recall=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum();return float(2*precision*recall/max(precision+recall,1e-12))
def score(clean,recon,region):
 if not region.sum():return None
 diff=clean.astype(np.float32)-recon.astype(np.float32);_,sm=ssim(clean,recon,data_range=255,full=True)
 return {'pixels':int(region.sum()),'mae':float(np.abs(diff[region]).mean()),'ssim':float(sm[region].mean()),'ink_f1':tol_f1(printed_ink(clean),printed_ink(recon),region),'edge_f1_tolerant':tol_f1(edges(clean),edges(recon),region),'dark_smudge_rate':float(np.sum((recon<180)&(clean>=220)&region)/max(np.sum((clean>=220)&region),1))}
rows=[]
for page in pages:
 clean=cv2.imread(str(clean_map[page]),0);record=next(x for x in cache_rows if x['page']==page)
 with np.load(record['cache']) as z:mask=z['mask'];outputs={m:z[m] for m in METHODS}
 regions=classify(clean,mask)
 for method,recon in outputs.items():
  for name,region in regions.items():
   q=score(clean,recon,region)
   if q:rows.append({'page':page,'method':method,'region':name,**q})
results=pd.DataFrame(rows);results.to_csv(TABLES/'metrics_by_page_region.csv',index=False)
results.groupby(['method','region'],as_index=False).mean(numeric_only=True).to_csv(TABLES/'summary_by_region.csv',index=False)


## Visual routing maps


In [ ]:
for page in pages[:24]:
 observed=cv2.imread(str(observed_map[page]),0);record=next(x for x in cache_rows if x['page']==page)
 with np.load(record['cache']) as z:
  for t in RAW_THRESHOLDS:
   rgb=cv2.cvtColor(observed,cv2.COLOR_GRAY2RGB);overlay=rgb.copy();overlay[z[f'g{t}_preserve']>0]=(20,160,20);overlay[z[f'g{t}_remove']>0]=(30,100,230);overlay[z[f'g{t}_ambiguous']>0]=(230,140,20)
   blend=cv2.addWeighted(rgb,.55,overlay,.45,0);cv2.imwrite(str(MAPS/f'{page}_geometry_t{t}_routes.png'),cv2.cvtColor(blend,cv2.COLOR_RGB2BGR))
print('Routing maps:',MAPS)


## Manifest


In [ ]:
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.5','schema':'exact_geometry_with_surrounding_evidence_v1','pages':len(pages),'excluded_pages':sorted(EXCLUDE_PAGES),'methods':METHODS,
'evidence_fields':list(evidence.columns),'guarantees':{'clean_used_only_after_reconstruction':True,'outside_mask_immutable':True,'preserve_route_byte_exact':True,'telea_only_on_ambiguous_route':True,'no_combined_confidence_score':True,'no_auto_acceptance':True},
'future_framework':{'axis_1':'annotation confidence, later supplied by YOLO and related evidence','axis_2':'correction safety, later derived from validated surrounding-area evidence'},
'limitations':['Genuine overlap is not directly observable from an annotated-only page.','Evidence variables are recorded, not calibrated.','Page201 is excluded because its historical mask/cache is known to be incorrect or stale.']}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2));print('COMPLETE:',OUT)
